# Static ASL sign recognition: controlled experiments

Static (single-image) recognition of 36 classes (digits 0-9, letters A-Z) on the **ASL-HG** dataset
([Mendeley Data](https://data.mendeley.com/datasets/j4y5w2c8w9/1), `ASL_Raw_Images.zip`). The dataset is not
included in this repo.

**Design principles**
- **Participant-independent split:** train P1-P8, validation P9, test P10. No person appears in two splits.
- **One variable at a time** against a fixed baseline CNN (same optimizer, batch size, early stopping).
- **Several seeds per experiment,** because single runs of the same code varied a lot.
- **Test set (P10) used once,** for the final chosen model only.

**Safe to Run All:** every training loop sits behind `RUN_TRAINING = False`. With the flag off, the notebook
loads results from `experiments/*.json` instead of retraining. `save_run` also refuses to overwrite a saved run.

**Code layout.** Plain-Python helpers (class list, participant split) are in `src/data.py`. The model, augmentation,
tf.data pipeline and training/bookkeeping code stay in this notebook, because they need TensorFlow and a GPU to verify.

## 0. Setup

In [ ]:
RUN_TRAINING = False   # True retrains the experiments below (hours on a laptop GPU)
RUN_P10_TEST = True    # evaluates the published final model on the held-out test participant

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import cv2
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Edit PROJECT_ROOT to match your machine. The dataset is expected at dataset/asl_dataset/<class>/P<k>_<class>_<id>.jpg
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_ROOT / "dataset" / "asl_dataset"
MODELS_DIR = PROJECT_ROOT / "models"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
MODELS_DIR.mkdir(exist_ok=True)
EXPERIMENTS_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 30            # ceiling; early stopping (patience 5 on val_loss) normally stops sooner

TRAIN_PARTICIPANTS = [f"P{i}" for i in range(1, 9)]   # P1-P8
VAL_PARTICIPANTS = ["P9"]
TEST_PARTICIPANTS = ["P10"]

DATASET_AVAILABLE = DATASET_DIR.exists()
print("Dataset found:", DATASET_AVAILABLE)

**Classes and split.** The class index is the position in the sorted folder list (`0`-`9` then `A`-`Z`). The same
order must be used for training, evaluation and the webcam app. Participants are read from the filename
(`P5_A_433.jpg` -> `P5`), so assigning images to splits needs no image loading. The class list and the split logic live in `src/data.py` (unit-tested in `tests/`).

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))   # makes the repo's `src/` package importable from notebooks/
from src.data import list_classes, collect_split as _collect_split

classes = list_classes(DATASET_DIR)   # sorted folder names if the dataset is present, else the canonical 0-9, A-Z list
class_to_index = {name: i for i, name in enumerate(classes)}

def collect_split(participants):
    return _collect_split(DATASET_DIR, classes, participants)

if DATASET_AVAILABLE:
    train_paths, train_labels = collect_split(TRAIN_PARTICIPANTS)
    val_paths, val_labels = collect_split(VAL_PARTICIPANTS)
    test_paths, test_labels = collect_split(TEST_PARTICIPANTS)
    print("Train / val / test:", len(train_paths), len(val_paths), len(test_paths))   # 28800 / 3600 / 3600
    assert not (set(train_paths) & set(val_paths)) and not (set(train_paths) & set(test_paths)) \
        and not (set(val_paths) & set(test_paths)), "split leakage"

**Input pipeline.** Baseline preprocessing is deliberately minimal: decode RGB, resize to 128x128, scale to 0-1.
The training set is shuffled **before** decoding, while each item is still just a path string. A full-size shuffle
buffer is then cheap (a buffer of decoded images would need ~5.6 GB). Validation and test stay unshuffled so
predictions line up with the label lists when building confusion matrices.

In [ ]:
def make_dataset(paths, labels, training, seed=SEED, image_size=IMAGE_SIZE):
    def load(image_path, label):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, image_size) / 255.0
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

if DATASET_AVAILABLE:
    train_dataset = make_dataset(train_paths, train_labels, training=True)
    val_dataset = make_dataset(val_paths, val_labels, training=False)
    test_dataset = make_dataset(test_paths, test_labels, training=False)

    images, labels = next(iter(train_dataset))
    print("Batch:", images.shape, labels.shape, "| pixel range:", float(images.numpy().min()), float(images.numpy().max()))

    plt.figure(figsize=(12, 6))
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(f"Class: {classes[labels[i].numpy()]}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1. Model and experiment helpers

**Baseline CNN** (about 426k parameters): four blocks of `Conv2D(3x3, ReLU, same padding)` + `MaxPooling2D(2x2)`
with 32, 64, 128 and 256 filters, then **global average pooling** (instead of Flatten: far fewer parameters, which
helps generalisation to an unseen person), `Dense(128, ReLU)`, `Dropout(0.3)` and `Dense(36, softmax)`.

Augmentation layers sit inside the model, so they are active only during training and do nothing at prediction time.

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
    return layers.MaxPooling2D((2, 2))(x)


@keras.utils.register_keras_serializable(package="signlang")
class ToGrayscale(layers.Layer):
    # RGB -> grayscale, repeated to 3 channels so the rest of the model is unchanged.
    def call(self, x):
        return tf.repeat(tf.image.rgb_to_grayscale(x), 3, axis=-1)


def build_cnn(name, image_size=IMAGE_SIZE, augmentation=None, grayscale=False):
    inputs = keras.Input(shape=(*image_size, 3))
    x = inputs
    if augmentation is not None:
        x = augmentation(x)
    if grayscale:
        x = ToGrayscale()(x)
    for filters in (32, 64, 128, 256):
        x = conv_block(x, filters)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(classes), activation="softmax")(x)
    return keras.Model(inputs, outputs, name=name)


print("Baseline parameters:", build_cnn("check").count_params())   # 425,956

In [ ]:
def make_augmentation():
    # Geometric + mild lighting changes. No horizontal flip: a flipped right hand is a left hand.
    return keras.Sequential([
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.15),
        layers.RandomRotation(0.04),
        layers.RandomBrightness(0.2, value_range=(0, 1)),
        layers.RandomContrast(0.2),
    ], name="augmentation")


class WebcamStyleAugment(layers.Layer):
    # Blur, sensor noise, saturation and per-channel colour gain. Active only in training.
    def call(self, x, training=None):
        if not training:
            return x
        b = tf.shape(x)[0]
        gray = tf.reduce_mean(x, axis=-1, keepdims=True)
        x = gray + (x - gray) * tf.random.uniform([b, 1, 1, 1], 0.3, 1.3)
        x = x * tf.random.uniform([b, 1, 1, 3], 0.9, 1.1)

        def blur():
            idx = tf.random.uniform([], 0, 3, dtype=tf.int32)
            return tf.switch_case(idx, [
                lambda: tf.nn.avg_pool2d(x, ksize=3, strides=1, padding="SAME"),
                lambda: tf.nn.avg_pool2d(x, ksize=5, strides=1, padding="SAME"),
                lambda: tf.nn.avg_pool2d(x, ksize=7, strides=1, padding="SAME"),
            ])

        x = tf.cond(tf.random.uniform([]) < 0.5, blur, lambda: x)
        x = x + tf.random.normal(tf.shape(x)) * tf.random.uniform([b, 1, 1, 1], 0.0, 0.05)
        return tf.clip_by_value(x, 0.0, 1.0)


def make_strong_augmentation():
    return keras.Sequential([
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.15),
        layers.RandomRotation(0.04),
        layers.RandomBrightness(0.3, value_range=(0, 1)),
        layers.RandomContrast(0.35),
        WebcamStyleAugment(),
    ], name="strong_augmentation")

**Experiment bookkeeping.** `save_run` stores each run under a unique name (`<experiment>_seed<k>`) with its
settings and full per-epoch history in `experiments/`, and refuses to overwrite. This was added after a "Run All"
silently overwrote an earlier baseline run (that run scored 70.3% on P9 and is no longer on disk).

In [ ]:
def save_run(name, seed, model, hist, notes="", image_size=IMAGE_SIZE):
    run_id = f"{name}_seed{seed}"
    model_path = MODELS_DIR / f"{run_id}.keras"
    json_path = EXPERIMENTS_DIR / f"{run_id}.json"
    if model_path.exists() or json_path.exists():
        raise FileExistsError(f"{run_id} already saved; pick a new name or seed.")
    best_epoch = int(np.argmin(hist["val_loss"])) + 1
    record = {
        "run_id": run_id, "seed": seed, "image_size": list(image_size), "batch_size": BATCH_SIZE,
        "train_participants": TRAIN_PARTICIPANTS, "val_participants": VAL_PARTICIPANTS,
        "best_epoch": best_epoch, "best_val_loss": float(min(hist["val_loss"])),
        "val_accuracy_at_best_epoch": float(hist["val_accuracy"][best_epoch - 1]),
        "history": {k: [float(v) for v in vals] for k, vals in hist.items()},
        "notes": notes,
    }
    model.save(model_path)
    json_path.write_text(json.dumps(record, indent=2))
    print("Saved:", model_path.name, "and", json_path.name)


def run_experiment(name, seeds, augmentation_fn=None, grayscale=False, image_size=IMAGE_SIZE, notes=""):
    # Same optimizer, loss, batch size and early stopping for every experiment; only the named variable changes.
    ds_val = make_dataset(val_paths, val_labels, training=False, image_size=image_size)
    for seed in seeds:
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        ds_train = make_dataset(train_paths, train_labels, training=True, seed=seed, image_size=image_size)
        model = build_cnn(f"{name}_cnn", image_size,
                          augmentation_fn() if augmentation_fn else None, grayscale)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                      loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
        h = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS, verbose=2,
                      callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                                               restore_best_weights=True)])
        save_run(name, seed, model, h.history, notes=notes, image_size=image_size)


def summarize(name):
    accs = []
    for path in sorted(EXPERIMENTS_DIR.glob(f"{name}_seed*.json")):
        r = json.loads(path.read_text())
        accs.append(r["val_accuracy_at_best_epoch"])
        print(f"{name:>10} seed {r['seed']}: best epoch {r['best_epoch']:>2}, "
              f"val_loss {r['best_val_loss']:.3f}, val_acc {r['val_accuracy_at_best_epoch']:.3f}")
    if accs:
        print(f"{name:>10} mean {np.mean(accs):.3f}  min {min(accs):.3f}  max {max(accs):.3f}\n")
    return accs


def plot_run(run_id):
    h = json.loads((EXPERIMENTS_DIR / f"{run_id}.json").read_text())["history"]
    ep = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a, key, title in ((ax[0], "loss", "Loss"), (ax[1], "accuracy", "Accuracy")):
        a.plot(ep, h[key], label="train")
        a.plot(ep, h["val_" + key], label="validation (P9)")
        a.set_title(f"{title}: {run_id}")
        a.set_xlabel("Epoch")
        a.legend()
    plt.tight_layout()
    plt.show()

## 2. Baseline (raw RGB, 128x128, no augmentation)

Three seeds. The seed changes weight initialisation and training order.

**Finding:** the same code gave 0.859, 0.765 and 0.630 on P9 (plus 0.703 on an earlier run that was overwritten).
A single baseline run says very little. Train accuracy reached about 97% while validation on the unseen participant
lagged far behind: the model learns the training participants well but does not transfer to a new person.

In [ ]:
if RUN_TRAINING and DATASET_AVAILABLE:
    run_experiment("baseline", [42, 43, 44], notes="raw RGB 128x128")
baseline_accs = summarize("baseline")
plot_run("baseline_seed42")

**What the baseline's errors look like (seed-42 run on P9, from the raw lab notebook).** Errors were all-or-nothing
per class: whole signs flipped to one wrong label for the whole participant (for example D -> R, F -> B, M -> E,
6 -> W), instead of scattered mistakes. Each class has only 100 images per participant, so one class flipping moves
accuracy by 2.8 points. Some pairs are close or identical handshapes in ASL (6/W, 2/V, M/E, R/U).

## 3. Augmentation (geometric + mild brightness/contrast)

Translate +/-10%, zoom +/-15%, rotate about +/-14 degrees, brightness +/-20%, contrast +/-20%. Hypothesis: the
model overfits to person-specific framing and lighting, so varying these in training should help on an unseen
participant.

**Finding:** mean P9 accuracy 0.923 versus 0.751 for the baseline, and far less variation between seeds
(0.908-0.932 versus 0.630-0.859). The worst augmented run beat the best baseline run.
Caveat: the seed-42 runs used a fixed shuffle seed; seeds 43 and 44 also varied the shuffle order.

In [ ]:
if RUN_TRAINING and DATASET_AVAILABLE:
    run_experiment("augmented", [42, 43, 44], augmentation_fn=make_augmentation,
                   notes="aug: translate .1, zoom .15, rotate .04, brightness .2, contrast .2")
augmented_accs = summarize("augmented")
plot_run("augmented_seed43")

## 4. Final model on the held-out test participant (P10)

Chosen by lowest validation loss among the augmented runs (seed 43), before looking at P10. P10 is evaluated once.

**Result: 83.25% on P10** (versus about 93% on P9, which also helped select the model). Most classes reach 98-100% recall;
the errors are whole classes absorbed by a look-alike: V -> 2, R -> U, 9 -> 8, E -> M, W -> 6 and others. The set
of failing classes differs from P9's, so a single held-out person is a noisy estimate. Rotating the held-out
participant is future work.

In [ ]:
if RUN_P10_TEST and DATASET_AVAILABLE and (MODELS_DIR / "augmented_seed43.keras").exists():
    final_model = keras.models.load_model(MODELS_DIR / "augmented_seed43.keras")
    test_pred = np.argmax(final_model.predict(test_dataset), axis=1)
    test_true = np.array(test_labels)   # valid because test_dataset is not shuffled
    print("Test accuracy (P10):", (test_pred == test_true).mean())
    print(classification_report(test_true, test_pred, target_names=classes, digits=3, zero_division=0))
    fig, ax = plt.subplots(figsize=(12, 12))
    ConfusionMatrixDisplay(confusion_matrix(test_true, test_pred), display_labels=classes).plot(
        ax=ax, colorbar=False, cmap="Blues")
    plt.title("Augmented CNN (seed 43): test (P10)")
    plt.tight_layout()
    plt.show()

## 5. Higher resolution (192x192): no benefit

Only the input size changes (the parameter count stays the same because of global average pooling). One seed was run.

**Finding:** seed 42 reached 0.893 on P9 versus 0.929 at 128x128, with about 2x the time per epoch. A per-class
check showed 192 fixed part of 8 but broke D, S, T and N. No evidence that fine detail was the bottleneck.
This is one seed, so the claim is "no evidence of benefit", not "proven worse".

In [ ]:
if RUN_TRAINING and DATASET_AVAILABLE:
    run_experiment("aug192", [42], augmentation_fn=make_augmentation, image_size=(192, 192),
                   notes="192x192 + augmentation")
summarize("aug192");

## 6. Strong augmentation (blur, noise, saturation, colour casts)

Motivation: a first live webcam test failed on a cluttered background with a soft, low-contrast, colour-cast
camera image, and the model was brittle to small contrast/colour changes. This adds blur, Gaussian noise,
saturation scaling (0.3-1.3), per-channel colour gain (+/-10%) and wider brightness/contrast ranges.

**Finding:** P9 accuracy dropped to a mean of 0.873 (0.885, 0.837, 0.896): harder training images cost a few
points on clean phone photos. P9 cannot show whether it helps a webcam, so the live test is the judge.

In [ ]:
if RUN_TRAINING and DATASET_AVAILABLE:
    run_experiment("augstrong", [42, 43, 44], augmentation_fn=make_strong_augmentation,
                   notes="strong aug: blur, noise, saturation, color gain")
augstrong_accs = summarize("augstrong")

## 7. Grayscale (layer inside the model)

Hypothesis: removing colour removes a source of mismatch between the dataset phones and a washed-out webcam.
Regular augmentation plus a `ToGrayscale` layer (same parameter count).

**Finding:** mean P9 accuracy 0.918 (0.904, 0.953, 0.898), about equal to regular augmentation (0.923) and noisier.
Not tested live; dropped for simplicity. Note: one seed (0.953) was the best single P9 run of the project,
but with three seeds that is more likely luck than a real improvement.

In [ ]:
if RUN_TRAINING and DATASET_AVAILABLE:
    run_experiment("auggray", [42, 43, 44], augmentation_fn=make_augmentation, grayscale=True,
                   notes="regular aug + grayscale layer")
auggray_accs = summarize("auggray")

## 8. Segmentation check: grayscale + median filter + Otsu (not adopted)

Looked at the masks before training anything. On P9 images (plain white backdrop) Otsu gives clean silhouettes, but
silhouettes erase interior detail (finger crossings in R, thumb contact in F), which is exactly where the model
already struggles. On webcam frames the dark door and shadows merged with the hand. Not pursued. A
skin-colour mask that keeps the hand's own pixels is a possible next experiment.

## 9. Summary

| Experiment | Seeds | P9 val accuracy (mean, range) |
|---|---|---|
| Baseline (raw RGB 128) | 3 | 0.751 (0.630-0.859) |
| + augmentation | 3 | 0.923 (0.908-0.932) |
| + augmentation, 192x192 | 1 | 0.893 |
| + strong augmentation | 3 | 0.873 (0.837-0.896) |
| + augmentation + grayscale | 3 | 0.918 (0.898-0.953) |

Final test (P10, one model, evaluated once): **83.25%**.

**Live webcam check (informal).** `augstrong_seed44` with a plain background and a 0.45 confidence threshold got
A, L, 5, O and Y right. That is a handful of signs, one person, one camera, one background: a demo, not an accuracy figure.
On a cluttered background the same model got 1 of 3 test frames right, and the models trained without the
webcam-style augmentation answered "H" for most signs.